# Phase 6 — Decision Engine

In this phase, we combine replenishment decisions and simulation results to generate clear product-level recommendations.

The goal is to transform technical inventory and simulation indicators into actionable business decisions.

## 6.1 Load Decision and Simulation Results

We load:
- Replenishment decisions from Phase 4
- Product-level simulation KPIs from Phase 5

These datasets will be combined using `product_id`.

In [1]:
import pandas as pd
import numpy as np

replenishment_decisions = pd.read_csv(
    "../data/replenishment_decisions.csv"
)

simulation_summary = pd.read_csv(
    "../data/product_simulation_summary.csv"
)

replenishment_decisions.head()

,product_id,current_stock,reorder_point,max_stock,avg_daily_forecast,days_of_supply,inventory_risk,needs_reorder,forecast_based_reorder_qty,replenishment_priority
0,P002,8,10,40,18.99,0.42,High Risk,True,32,Critical
1,P006,3,5,15,7.13,0.42,High Risk,True,12,Critical
2,P010,6,8,30,11.74,0.51,High Risk,True,24,Critical
3,P004,4,5,20,4.33,0.92,High Risk,True,16,Critical
4,P003,15,8,35,11.28,1.33,High Risk,True,20,High


In [2]:
print("Replenishment shape:", replenishment_decisions.shape)
print("Simulation summary shape:", simulation_summary.shape)

print("\nReplenishment columns:")
print(replenishment_decisions.columns.tolist())

print("\nSimulation columns:")
print(simulation_summary.columns.tolist())

print("\nMissing values:")
print(replenishment_decisions.isna().sum())
print(simulation_summary.isna().sum())

Replenishment shape: (10, 10)
Simulation summary shape: (10, 6)

Replenishment columns:
['product_id', 'current_stock', 'reorder_point', 'max_stock', 'avg_daily_forecast', 'days_of_supply', 'inventory_risk', 'needs_reorder', 'forecast_based_reorder_qty', 'replenishment_priority']

Simulation columns:
['product_id', 'total_demand', 'fulfilled_demand', 'stockout_qty', 'stockout_days', 'service_level']

Missing values:
product_id                    0
current_stock                 0
reorder_point                 0
max_stock                     0
avg_daily_forecast            0
days_of_supply                0
inventory_risk                0
needs_reorder                 0
forecast_based_reorder_qty    0
replenishment_priority        0
dtype: int64
product_id          0
total_demand        0
fulfilled_demand    0
stockout_qty        0
stockout_days       0
service_level       0
dtype: int64


## 6.2 Combine Replenishment and Simulation Results

To generate final recommendations, we combine the replenishment decisions from Phase 4 with the simulation KPIs from Phase 5.

The datasets are merged using `product_id`.

This creates one decision table containing both:
- Inventory and replenishment indicators
- Simulation performance indicators

In [3]:
decision_engine = replenishment_decisions.merge(
    simulation_summary,
    on="product_id",
    how="left"
)

decision_engine

,product_id,current_stock,reorder_point,max_stock,avg_daily_forecast,days_of_supply,inventory_risk,needs_reorder,forecast_based_reorder_qty,replenishment_priority,total_demand,fulfilled_demand,stockout_qty,stockout_days,service_level
0,P002,8,10,40,18.99,0.42,High Risk,True,32,Critical,265,40,225,13,15.09
1,P006,3,5,15,7.13,0.42,High Risk,True,12,Critical,99,15,84,12,15.15
2,P010,6,8,30,11.74,0.51,High Risk,True,24,Critical,163,30,133,12,18.40
3,P004,4,5,20,4.33,0.92,High Risk,True,16,Critical,62,20,42,10,32.26
4,P003,15,8,35,11.28,1.33,High Risk,True,20,High,158,35,123,12,22.15
5,P008,25,8,40,14.26,1.75,High Risk,True,15,High,201,40,161,12,19.90
6,P001,12,5,30,6.49,1.85,High Risk,True,18,High,91,30,61,10,32.97
7,P005,20,6,30,9.21,2.17,Medium Risk,True,10,Medium,128,30,98,11,23.44
8,P007,7,4,12,2.68,2.61,Medium Risk,True,5,Medium,37,12,25,10,32.43
9,P009,18,5,25,5.78,3.11,Medium Risk,True,7,Medium,82,25,57,10,30.49


In [4]:
print("Decision engine shape:", decision_engine.shape)

decision_engine[
    [
        "product_id",
        "days_of_supply",
        "replenishment_priority",
        "forecast_based_reorder_qty",
        "stockout_qty",
        "stockout_days",
        "service_level"
    ]
]

Decision engine shape: (10, 15)


,product_id,days_of_supply,replenishment_priority,forecast_based_reorder_qty,stockout_qty,stockout_days,service_level
0,P002,0.42,Critical,32,225,13,15.09
1,P006,0.42,Critical,12,84,12,15.15
2,P010,0.51,Critical,24,133,12,18.40
3,P004,0.92,Critical,16,42,10,32.26
4,P003,1.33,High,20,123,12,22.15
5,P008,1.75,High,15,161,12,19.90
6,P001,1.85,High,18,61,10,32.97
7,P005,2.17,Medium,10,98,11,23.44
8,P007,2.61,Medium,5,25,10,32.43
9,P009,3.11,Medium,7,57,10,30.49


## 6.3 Generate Final Product Recommendations

The decision engine converts inventory and simulation indicators into clear business actions.

The recommendation logic considers:
- Replenishment priority
- Service level
- Stockout days
- Reorder requirement

The objective is to produce a simple and actionable recommendation for each product.

In [5]:
def generate_recommendation(row):

    if row["replenishment_priority"] == "Critical":
        return "Replenish Immediately"

    elif (
        row["replenishment_priority"] == "High"
        and row["service_level"] < 50
    ):
        return "Increase Reorder Quantity"

    elif row["replenishment_priority"] == "Medium":
        return "Monitor and Replenish Soon"

    elif row["needs_reorder"]:
        return "Replenish"

    else:
        return "Maintain Current Policy"


decision_engine["recommendation"] = (
    decision_engine.apply(
        generate_recommendation,
        axis=1
    )
)

In [6]:
decision_engine[
    [
        "product_id",
        "replenishment_priority",
        "service_level",
        "stockout_days",
        "needs_reorder",
        "recommendation"
    ]
]

,product_id,replenishment_priority,service_level,stockout_days,needs_reorder,recommendation
0,P002,Critical,15.09,13,True,Replenish Immediately
1,P006,Critical,15.15,12,True,Replenish Immediately
2,P010,Critical,18.40,12,True,Replenish Immediately
3,P004,Critical,32.26,10,True,Replenish Immediately
4,P003,High,22.15,12,True,Increase Reorder Quantity
5,P008,High,19.90,12,True,Increase Reorder Quantity
6,P001,High,32.97,10,True,Increase Reorder Quantity
7,P005,Medium,23.44,11,True,Monitor and Replenish Soon
8,P007,Medium,32.43,10,True,Monitor and Replenish Soon
9,P009,Medium,30.49,10,True,Monitor and Replenish Soon


## 6.4 Rank Products by Decision Urgency

To prioritize warehouse actions, we assign a numerical severity score to each recommendation.

The score allows products to be sorted from the most urgent action to the least urgent.

Severity mapping:
- Replenish Immediately → 0
- Increase Reorder Quantity → 1
- Monitor and Replenish Soon → 2
- Replenish → 3
- Maintain Current Policy → 4

In [7]:
severity_order = {
    "Replenish Immediately": 0,
    "Increase Reorder Quantity": 1,
    "Monitor and Replenish Soon": 2,
    "Replenish": 3,
    "Maintain Current Policy": 4
}

decision_engine["severity_rank"] = (
    decision_engine["recommendation"]
    .map(severity_order)
)

In [8]:
final_decisions = (
    decision_engine[
        [
            "product_id",
            "current_stock",
            "days_of_supply",
            "replenishment_priority",
            "service_level",
            "stockout_days",
            "stockout_qty",
            "forecast_based_reorder_qty",
            "recommendation",
            "severity_rank"
        ]
    ]
    .sort_values(
        [
            "severity_rank",
            "service_level"
        ],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

final_decisions

,product_id,current_stock,days_of_supply,replenishment_priority,service_level,stockout_days,stockout_qty,forecast_based_reorder_qty,recommendation,severity_rank
0,P002,8,0.42,Critical,15.09,13,225,32,Replenish Immediately,0
1,P006,3,0.42,Critical,15.15,12,84,12,Replenish Immediately,0
2,P010,6,0.51,Critical,18.40,12,133,24,Replenish Immediately,0
3,P004,4,0.92,Critical,32.26,10,42,16,Replenish Immediately,0
4,P008,25,1.75,High,19.90,12,161,15,Increase Reorder Quantity,1
5,P003,15,1.33,High,22.15,12,123,20,Increase Reorder Quantity,1
6,P001,12,1.85,High,32.97,10,61,18,Increase Reorder Quantity,1
7,P005,20,2.17,Medium,23.44,11,98,10,Monitor and Replenish Soon,2
8,P009,18,3.11,Medium,30.49,10,57,7,Monitor and Replenish Soon,2
9,P007,7,2.61,Medium,32.43,10,25,5,Monitor and Replenish Soon,2


## 6.5 Build a Final Decision Summary

To make the decision engine easier to review, we summarize the number of products assigned to each recommendation.

This provides a quick overview of how many products require immediate action, stronger replenishment, or monitoring.

In [9]:
recommendation_summary = (
    final_decisions["recommendation"]
    .value_counts()
    .reset_index()
)

recommendation_summary.columns = [
    "recommendation",
    "product_count"
]

recommendation_summary

,recommendation,product_count
0,Replenish Immediately,4
1,Increase Reorder Quantity,3
2,Monitor and Replenish Soon,3


In [10]:
top_priority_products = final_decisions.head(5)

top_priority_products[
    [
        "product_id",
        "service_level",
        "stockout_days",
        "recommendation"
    ]
]

,product_id,service_level,stockout_days,recommendation
0,P002,15.09,13,Replenish Immediately
1,P006,15.15,12,Replenish Immediately
2,P010,18.40,12,Replenish Immediately
3,P004,32.26,10,Replenish Immediately
4,P008,19.90,12,Increase Reorder Quantity


## 6.6 Save Final Decision Engine Output

The final decision-engine outputs are saved for use in the final dashboard.

We save:
- Final product-level decisions
- Recommendation summary
- Top-priority products

In [11]:
final_decisions.to_csv(
    "../data/final_decisions.csv",
    index=False
)

recommendation_summary.to_csv(
    "../data/recommendation_summary.csv",
    index=False
)

top_priority_products.to_csv(
    "../data/top_priority_products.csv",
    index=False
)

print("Decision engine outputs saved successfully.")

Decision engine outputs saved successfully.


## Phase 6 Conclusion

Phase 6 combined replenishment decisions and simulation results into a final product-level decision engine.

The system uses:
- Inventory coverage
- Replenishment priority
- Service level
- Stockout quantity
- Stockout days
- Reorder requirements

to generate clear operational recommendations.

The final recommendations include:
- Replenish Immediately
- Increase Reorder Quantity
- Monitor and Replenish Soon
- Replenish
- Maintain Current Policy

Products are ranked by urgency so that the most critical inventory actions appear first.

The next phase will present the forecasting, replenishment, simulation, and decision-engine outputs in a final dashboard/application.